# Modulo 5 · Lezione 3 — FastAPI di base

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento | |
|---|------|-|
| 1 | Cos'è FastAPI? | Framework moderno, veloce, con tipizzazione |
| 2 | Installazione e primo server | `uvicorn`, `app = FastAPI()` |
| 3 | Path params | `/items/{id}` |
| 4 | Query params | `/items?skip=0&limit=10` |
| 5 | Pydantic — validazione | Modelli di dati con tipizzazione |
| 6 | Request body | POST / PUT con body JSON |
| 7 | Response models | Controllare cosa restituisce l'API |
| 8 | Gestione degli errori | `HTTPException` |
| 9 | Pratica | API di task in memoria |

> ⚠️ FastAPI ha bisogno di un server ASGI (`uvicorn`) per essere eseguito. In questo notebook lo lanciamo in un thread in background e lo chiamiamo con `requests` dalla stessa sessione.

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'requests', 'httpx']:
    try:
        nombre = pkg.split('[')[0]
        __import__(nombre)
        print(f"{nombre}: già installato")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: installato")

---
## 1. Cos'è FastAPI?

FastAPI è un framework Python moderno per costruire API:

```
VANTAGGI:
✦ Molto veloce — paragonabile a Node.js e Go (grazie a Starlette + uvicorn)
✦ Tipizzato — usa i type hint di Python per validare automaticamente
✦ Documentazione automatica — Swagger UI e ReDoc generati da soli
✦ Meno codice — meno bug, meno boilerplate
✦ Async nativo — supporta async/await fin dall'inizio

STACK TIPICO:
FastAPI   ← framework (definisce rotte, validazione, serializzazione)
Pydantic  ← validazione dei dati (incluso con FastAPI)
Uvicorn   ← server ASGI che esegue l'app
SQLAlchemy← ORM per il database (Lezione 4)

CONFRONTO:
Flask    → minimalista, senza validazione, senza async nativo
Django   → completo ma pesante, più orientato a MVC/web
FastAPI  → equilibrio: veloce, validazione, async, ideale per le API
```

---
## 2. Installazione e primo server

In produzione eseguiresti:
```bash
uvicorn main:app --host 0.0.0.0 --port 8000 --reload
```

Nel notebook lanciamo il server in un thread in background.

In [ ]:
from fastapi import FastAPI
import uvicorn, threading, time, requests

# ── Definire l'app ───────────────────────────────────────────
app1 = FastAPI(title="La mia prima API", version="0.1.0")

@app1.get("/")
def raiz():
    return {"mensaje": "Ciao da FastAPI!"}

@app1.get("/salud")
def salud():
    return {"status": "ok", "version": "0.1.0"}

# ── Lanciare in un thread ───────────────────────────────────────────
config = uvicorn.Config(app1, host="127.0.0.1", port=8001, log_level="warning")
server = uvicorn.Server(config)
hilo   = threading.Thread(target=server.run, daemon=True)
hilo.start()
time.sleep(1.5)  # aspettare che si avvii

# ── Provare ───────────────────────────────────────────────────
print("GET /")
print(requests.get("http://127.0.0.1:8001/").json())

print("\nGET /salud")
print(requests.get("http://127.0.0.1:8001/salud").json())

print("\n📖 Documentazione interattiva disponibile su:")
print("   http://127.0.0.1:8001/docs   (Swagger UI)")
print("   http://127.0.0.1:8001/redoc  (ReDoc)")

---
## 3. Path parameters

```python
@app.get("/items/{item_id}")
def get_item(item_id: int):   # FastAPI converte e valida automaticamente
    ...

# /items/42   → item_id = 42  (int)
# /items/abc  → 422 Unprocessable Entity (non è un int)
```

In [ ]:
from fastapi import FastAPI, Path
from enum import Enum

app2 = FastAPI(title="Path params demo")

# Dati simulati
PRODUCTOS = {
    1: {"id": 1, "nombre": "Tastiera",    "precio": 85.0,  "categoria": "informatica"},
    2: {"id": 2, "nombre": "Mouse",       "precio": 45.0,  "categoria": "informatica"},
    3: {"id": 3, "nombre": "Smartphone",  "precio": 599.0, "categoria": "elettronica"},
}

class Categoria(str, Enum):
    informatica = "informatica"
    electronica = "electronica"
    hogar       = "hogar"

@app2.get("/productos")
def listar_productos():
    return list(PRODUCTOS.values())

@app2.get("/productos/{producto_id}")
def obtener_producto(
    producto_id: int = Path(description="ID del prodotto", ge=1)
    # ge=1 → greater or equal 1
):
    if producto_id not in PRODUCTOS:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail=f"Prodotto {producto_id} non trovato")
    return PRODUCTOS[producto_id]

@app2.get("/categorias/{categoria}")
def productos_por_categoria(categoria: Categoria):  # Enum valida i valori possibili
    return [p for p in PRODUCTOS.values() if p['categoria'] == categoria.value]


# Lanciare
cfg2 = uvicorn.Config(app2, host="127.0.0.1", port=8002, log_level="warning")
srv2 = uvicorn.Server(cfg2)
threading.Thread(target=srv2.run, daemon=True).start()
time.sleep(1.2)

BASE = "http://127.0.0.1:8002"
print("GET /productos:")
print(requests.get(f"{BASE}/productos").json())

print("\nGET /productos/1:")
print(requests.get(f"{BASE}/productos/1").json())

print("\nGET /productos/99 (404):")
r = requests.get(f"{BASE}/productos/99")
print(f"  status={r.status_code}, detail={r.json()['detail']}")

print("\nGET /productos/abc (422):")
r = requests.get(f"{BASE}/productos/abc")
print(f"  status={r.status_code}")

print("\nGET /categorias/informatica:")
print(requests.get(f"{BASE}/categorias/informatica").json())

---
## 4. Query parameters

```python
@app.get("/items")
def listar(skip: int = 0, limit: int = 10, activo: bool = True):
    ...
# GET /items?skip=20&limit=5&activo=false

# Opzionale con None
def buscar(q: str | None = None):
    ...
# GET /buscar           → q = None
# GET /buscar?q=tastiera → q = "tastiera"
```

In [ ]:
from fastapi import FastAPI, Query
from typing import Optional

app3 = FastAPI()

@app3.get("/productos")
def listar(
    skip:       int            = Query(0,    ge=0, description="Elementi da saltare"),
    limit:      int            = Query(10,   ge=1, le=100, description="Numero massimo da restituire"),
    categoria:  Optional[str]  = Query(None, description="Filtra per categoria"),
    precio_max: Optional[float]= Query(None, gt=0, description="Prezzo massimo"),
    q:          Optional[str]  = Query(None, min_length=2, description="Ricerca per nome"),
):
    items = list(PRODUCTOS.values())

    if categoria:
        items = [p for p in items if p['categoria'] == categoria]
    if precio_max is not None:
        items = [p for p in items if p['precio'] <= precio_max]
    if q:
        items = [p for p in items if q.lower() in p['nombre'].lower()]

    total = len(items)
    items = items[skip : skip + limit]
    return {"total": total, "skip": skip, "limit": limit, "items": items}


cfg3 = uvicorn.Config(app3, host="127.0.0.1", port=8003, log_level="warning")
srv3 = uvicorn.Server(cfg3)
threading.Thread(target=srv3.run, daemon=True).start()
time.sleep(1.2)

BASE3 = "http://127.0.0.1:8003"

print("GET /productos (tutti):")
print(requests.get(f"{BASE3}/productos").json())

print("\nGET /productos?categoria=informatica:")
print(requests.get(f"{BASE3}/productos", params={"categoria": "informatica"}).json())

print("\nGET /productos?precio_max=100:")
print(requests.get(f"{BASE3}/productos", params={"precio_max": 100}).json())

print("\nGET /productos?q=a (422 se q < 2 caratteri):")
r = requests.get(f"{BASE3}/productos", params={"q": "a"})
print(f"  status={r.status_code}")

---
## 5 e 6. Pydantic e Request body

**Pydantic** definisce la struttura dei dati in ingresso e in uscita:

```python
from pydantic import BaseModel, Field

class Producto(BaseModel):
    nombre:   str   = Field(min_length=1, max_length=100)
    precio:   float = Field(gt=0)
    stock:    int   = Field(ge=0, default=0)
    activo:   bool  = True

# FastAPI analizza il JSON del body, valida rispetto al modello
# e restituisce automaticamente 422 in caso di errori
@app.post("/productos")
def crear(producto: Producto):
    return producto
```

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

app4 = FastAPI(title="API con Pydantic")

# Modelli Pydantic
class ProductoBase(BaseModel):
    nombre:    str   = Field(min_length=1, max_length=100, examples=["Tastiera meccanica"])
    precio:    float = Field(gt=0, description="Prezzo in euro")
    stock:     int   = Field(ge=0, default=0)
    categoria: str   = Field(min_length=1)

class ProductoCreate(ProductoBase):
    pass  # stesso modello per la creazione

class ProductoUpdate(BaseModel):
    nombre:    Optional[str]   = Field(None, min_length=1)
    precio:    Optional[float] = Field(None, gt=0)
    stock:     Optional[int]   = Field(None, ge=0)
    categoria: Optional[str]   = None

class ProductoResponse(ProductoBase):
    id: int

# Database in memoria
db: dict[int, dict] = {}
contador = 0

@app4.get("/productos", response_model=list[ProductoResponse])
def listar():
    return list(db.values())

@app4.get("/productos/{prod_id}", response_model=ProductoResponse)
def obtener(prod_id: int):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Prodotto non trovato")
    return db[prod_id]

@app4.post("/productos", response_model=ProductoResponse, status_code=201)
def crear(producto: ProductoCreate):
    global contador
    contador += 1
    nuevo = {"id": contador, **producto.model_dump()}
    db[contador] = nuevo
    return nuevo

@app4.put("/productos/{prod_id}", response_model=ProductoResponse)
def reemplazar(prod_id: int, producto: ProductoCreate):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Prodotto non trovato")
    db[prod_id] = {"id": prod_id, **producto.model_dump()}
    return db[prod_id]

@app4.patch("/productos/{prod_id}", response_model=ProductoResponse)
def actualizar(prod_id: int, cambios: ProductoUpdate):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Prodotto non trovato")
    datos = db[prod_id]
    for campo, valor in cambios.model_dump(exclude_none=True).items():
        datos[campo] = valor
    return datos

@app4.delete("/productos/{prod_id}", status_code=204)
def eliminar(prod_id: int):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Prodotto non trovato")
    del db[prod_id]


cfg4 = uvicorn.Config(app4, host="127.0.0.1", port=8004, log_level="warning")
srv4 = uvicorn.Server(cfg4)
threading.Thread(target=srv4.run, daemon=True).start()
time.sleep(1.2)

BASE4 = "http://127.0.0.1:8004"

# CRUD completo
print("=== POST (crea) ===")
p1 = requests.post(f"{BASE4}/productos",
                   json={"nombre": "Tastiera meccanica", "precio": 85.0, "stock": 50, "categoria": "informatica"})
print(f"  {p1.status_code}: {p1.json()}")

p2 = requests.post(f"{BASE4}/productos",
                   json={"nombre": "Mouse wireless", "precio": 45.0, "stock": 100, "categoria": "informatica"})

print("\n=== POST con dati non validi (422) ===")
r = requests.post(f"{BASE4}/productos",
                  json={"nombre": "", "precio": -10, "categoria": "x"})  # nome vuoto, prezzo negativo
print(f"  {r.status_code}: {len(r.json()['detail'])} errori di validazione")

print("\n=== GET lista ===")
print(requests.get(f"{BASE4}/productos").json())

print("\n=== PATCH (aggiorna prezzo) ===")
r = requests.patch(f"{BASE4}/productos/1", json={"precio": 79.0})
print(f"  {r.status_code}: {r.json()}")

print("\n=== DELETE ===")
r = requests.delete(f"{BASE4}/productos/2")
print(f"  {r.status_code}")
print(f"  Rimasti: {requests.get(f'{BASE4}/productos').json()}")

---
## 9. Pratica — API di task

---

### Esercizio 1 — Modello con validazione

**Consegna:** Definisci i modelli Pydantic per un'API di gestione dei task di un team:

- `TareaCreate`: titulo (1-200 caratteri), descripcion (opzionale), prioridad (`baja|normal|alta|urgente`), fecha_limite (stringa ISO opzionale), asignado_a (str opzionale)
- `TareaUpdate`: tutti i campi opzionali
- `TareaResponse`: id (int), creada_en (str), completada (bool), + tutti quelli di TareaCreate

Verifica che i modelli funzionino: crea istanze valide e non valide.

In [ ]:
# ✏️ IL TUO CODICE QUI
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
from enum import Enum
from datetime import datetime

class Prioridad(str, Enum):
    baja    = "baja"
    normal  = "normal"
    alta    = "alta"
    urgente = "urgente"

class TareaCreate(BaseModel):
    titulo:       str                   = Field(min_length=1, max_length=200)
    descripcion:  Optional[str]         = None
    prioridad:    Prioridad             = Prioridad.normal
    fecha_limite: Optional[str]         = None
    asignado_a:   Optional[str]         = None

class TareaUpdate(BaseModel):
    titulo:       Optional[str]         = Field(None, min_length=1, max_length=200)
    descripcion:  Optional[str]         = None
    prioridad:    Optional[Prioridad]   = None
    fecha_limite: Optional[str]         = None
    asignado_a:   Optional[str]         = None
    completada:   Optional[bool]        = None

class TareaResponse(TareaCreate):
    id:          int
    creada_en:   str
    completada:  bool = False

# Valida
t = TareaCreate(titulo="Revisar PR", prioridad="alta", asignado_a="Ana")
print("Valida:", t)

# Non valida: titolo vuoto
try:
    TareaCreate(titulo="", prioridad="normal")
except ValidationError as e:
    print(f"Errore validazione: {e.error_count()} errori")

# Non valida: priorità fuori dall'enum
try:
    TareaCreate(titulo="Test", prioridad="maxima")
except ValidationError as e:
    print(f"Errore enum: {e.errors()[0]['msg']}")
```
</details>

---
### Esercizio 2 — API completa dei task

**Consegna:** Implementa l'API completa dei task con tutti gli endpoint:

- `GET /tareas` — lista, con filtri opzionali: `?completada=true`, `?prioridad=alta`, `?asignado_a=Ana`
- `GET /tareas/{id}` — ottieni un task
- `POST /tareas` — crea
- `PATCH /tareas/{id}` — aggiorna campi
- `POST /tareas/{id}/completar` — segna come completato
- `DELETE /tareas/{id}` — elimina
- `GET /tareas/estadisticas` — totale, completati, in sospeso per priorità

In [ ]:
# ✏️ IL TUO CODICE QUI
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum
import uvicorn, threading, time, requests


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum
import uvicorn, threading, time, requests
from datetime import datetime

class Prioridad(str, Enum):
    baja = "baja"; normal = "normal"; alta = "alta"; urgente = "urgente"

class TareaCreate(BaseModel):
    titulo:       str             = Field(min_length=1, max_length=200)
    descripcion:  Optional[str]   = None
    prioridad:    Prioridad       = Prioridad.normal
    asignado_a:   Optional[str]   = None

class TareaUpdate(BaseModel):
    titulo:       Optional[str]       = Field(None, min_length=1)
    descripcion:  Optional[str]       = None
    prioridad:    Optional[Prioridad] = None
    asignado_a:   Optional[str]       = None
    completada:   Optional[bool]      = None

class TareaResponse(TareaCreate):
    id: int; creada_en: str; completada: bool = False


app_tareas = FastAPI(title="API dei Task")
bd: dict[int, dict] = {}
seq = 0

@app_tareas.get("/tareas", response_model=list[TareaResponse])
def listar(completada: Optional[bool] = None, prioridad: Optional[Prioridad] = None,
           asignado_a: Optional[str] = None):
    items = list(bd.values())
    if completada  is not None: items = [t for t in items if t['completada'] == completada]
    if prioridad:               items = [t for t in items if t['prioridad'] == prioridad]
    if asignado_a:              items = [t for t in items if t.get('asignado_a') == asignado_a]
    return items

@app_tareas.get("/tareas/estadisticas")
def estadisticas():
    todas = list(bd.values())
    return {
        "total":      len(todas),
        "completadas":sum(1 for t in todas if t['completada']),
        "pendientes": sum(1 for t in todas if not t['completada']),
        "por_prioridad": {p.value: sum(1 for t in todas if t['prioridad'] == p.value) for p in Prioridad}
    }

@app_tareas.get("/tareas/{id}", response_model=TareaResponse)
def obtener(id: int):
    if id not in bd: raise HTTPException(404, "Task non trovato")
    return bd[id]

@app_tareas.post("/tareas", response_model=TareaResponse, status_code=201)
def crear(t: TareaCreate):
    global seq; seq += 1
    nueva = {"id": seq, "creada_en": datetime.now().isoformat(), "completada": False, **t.model_dump()}
    bd[seq] = nueva; return nueva

@app_tareas.patch("/tareas/{id}", response_model=TareaResponse)
def actualizar(id: int, cambios: TareaUpdate):
    if id not in bd: raise HTTPException(404, "Task non trovato")
    for k, v in cambios.model_dump(exclude_none=True).items():
        bd[id][k] = v
    return bd[id]

@app_tareas.post("/tareas/{id}/completar", response_model=TareaResponse)
def completar(id: int):
    if id not in bd: raise HTTPException(404, "Task non trovato")
    bd[id]['completada'] = True; return bd[id]

@app_tareas.delete("/tareas/{id}", status_code=204)
def eliminar(id: int):
    if id not in bd: raise HTTPException(404, "Task non trovato")
    del bd[id]


cfg = uvicorn.Config(app_tareas, host="127.0.0.1", port=8005, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)

B = "http://127.0.0.1:8005"
requests.post(f"{B}/tareas", json={"titulo": "Revisar PR", "prioridad": "alta", "asignado_a": "Ana"})
requests.post(f"{B}/tareas", json={"titulo": "Daily standup", "prioridad": "normal"})
requests.post(f"{B}/tareas", json={"titulo": "Deploy a prod", "prioridad": "urgente", "asignado_a": "Ana"})
requests.post(f"{B}/tareas/{1}/completar")
print("Statistiche:", requests.get(f"{B}/tareas/estadisticas").json())
print("Task di Ana:", [t['titulo'] for t in requests.get(f"{B}/tareas", params={"asignado_a": "Ana"}).json()])
```
</details>

---
### Esercizio 3 — Errori personalizzati

**Consegna:** Aggiungi all'API dell'esercizio 2 le seguenti regole di business con errori HTTP appropriati:

1. Non si può completare un task già completato → 409 Conflict.
2. Non si può eliminare un task urgente → 403 Forbidden.
3. Il titolo deve essere univoco tra i task attivi → 409 Conflict.
4. Aggiungi un endpoint `GET /tareas/{id}/historial` che per ora restituisce un 501 Not Implemented.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
# Aggiungere le validazioni agli endpoint esistenti:

# In crear:
activos = [t for t in bd.values() if not t['completada']]
if any(t['titulo'].lower() == t_create.titulo.lower() for t in activos):
    raise HTTPException(status_code=409, detail="Esiste già un task attivo con questo titolo")

# In completar:
if bd[id]['completada']:
    raise HTTPException(status_code=409, detail="Il task è già completato")

# In eliminar:
if bd[id]['prioridad'] == 'urgente':
    raise HTTPException(status_code=403, detail="Non è possibile eliminare i task urgenti")

# Nuovo endpoint:
@app_tareas.get("/tareas/{id}/historial")
def historial(id: int):
    if id not in bd:
        raise HTTPException(status_code=404, detail="Task non trovato")
    raise HTTPException(status_code=501, detail="Storico non ancora implementato")
```
</details>

---
### Esercizio 4 — Sotto-risorse

**Consegna:** Estendi l'API con i commenti sui task:

- `GET  /tareas/{id}/comentarios` — elenca i commenti di un task
- `POST /tareas/{id}/comentarios` — aggiunge un commento
- Modello `Comentario`: id, texto (1-500 caratteri), autor, creado_en
- Se si tenta di commentare un task inesistente → 404

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
from pydantic import BaseModel, Field
from datetime import datetime

class ComentarioCreate(BaseModel):
    texto: str = Field(min_length=1, max_length=500)
    autor: str = Field(min_length=1)

class ComentarioResponse(ComentarioCreate):
    id: int
    tarea_id: int
    creado_en: str

comentarios: dict[int, list] = {}  # {tarea_id: [comentario, ...]}
seq_com = 0

@app_tareas.get("/tareas/{id}/comentarios", response_model=list[ComentarioResponse])
def listar_comentarios(id: int):
    if id not in bd:
        raise HTTPException(404, "Task non trovato")
    return comentarios.get(id, [])

@app_tareas.post("/tareas/{id}/comentarios", response_model=ComentarioResponse, status_code=201)
def crear_comentario(id: int, comentario: ComentarioCreate):
    global seq_com
    if id not in bd:
        raise HTTPException(404, "Task non trovato")
    seq_com += 1
    nuevo = {"id": seq_com, "tarea_id": id, "creado_en": datetime.now().isoformat(),
             **comentario.model_dump()}
    comentarios.setdefault(id, []).append(nuevo)
    return nuevo
```
</details>

---
### Esercizio 5 (sfida) — API di votazioni

**Consegna:** Implementa da zero un'API di votazioni per le decisioni di team:

- **Domande**: titulo, descripcion, opciones (list[str], minimo 2), cerrada (bool)
- `GET /preguntas` — elenca tutte
- `POST /preguntas` — crea una domanda
- `POST /preguntas/{id}/votar` — vota per un'opzione (`{"opcion": "Opzione A", "votante": "Ana"}`)
  - Validare che l'opzione esista
  - Validare che la domanda non sia chiusa
  - Un votante può votare una sola volta per domanda
- `POST /preguntas/{id}/cerrar` — chiude la votazione
- `GET /preguntas/{id}/resultado` — mostra i voti e l'opzione vincente

In [ ]:
# ✏️ IL TUO CODICE QUI
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests
from datetime import datetime

class PreguntaCreate(BaseModel):
    titulo:      str       = Field(min_length=1, max_length=200)
    descripcion: Optional[str] = None
    opciones:    list[str] = Field(min_length=2)

class VotoCreate(BaseModel):
    opcion:  str = Field(min_length=1)
    votante: str = Field(min_length=1)

app_votos = FastAPI(title="API Votazioni")
preguntas: dict[int, dict] = {}
votos: dict[int, list] = {}     # {pregunta_id: [{opcion, votante}]}
seq = 0

@app_votos.post("/preguntas", status_code=201)
def crear_pregunta(p: PreguntaCreate):
    global seq; seq += 1
    nueva = {"id": seq, "cerrada": False, "creada_en": datetime.now().isoformat(), **p.model_dump()}
    preguntas[seq] = nueva; votos[seq] = []
    return nueva

@app_votos.get("/preguntas")
def listar(): return list(preguntas.values())

@app_votos.post("/preguntas/{id}/votar")
def votar(id: int, voto: VotoCreate):
    if id not in preguntas: raise HTTPException(404, "Domanda non trovata")
    p = preguntas[id]
    if p['cerrada']:         raise HTTPException(409, "La votazione è chiusa")
    if voto.opcion not in p['opciones']: raise HTTPException(400, "Opzione non valida")
    ya_voto = any(v['votante'] == voto.votante for v in votos[id])
    if ya_voto: raise HTTPException(409, "Questo votante ha già votato")
    votos[id].append(voto.model_dump())
    return {"mensaje": "Voto registrato", "votos_totales": len(votos[id])}

@app_votos.post("/preguntas/{id}/cerrar")
def cerrar(id: int):
    if id not in preguntas: raise HTTPException(404, "Domanda non trovata")
    preguntas[id]['cerrada'] = True
    return {"mensaje": "Votazione chiusa"}

@app_votos.get("/preguntas/{id}/resultado")
def resultado(id: int):
    if id not in preguntas: raise HTTPException(404, "Domanda non trovata")
    p = preguntas[id]
    conteo = {op: sum(1 for v in votos[id] if v['opcion'] == op) for op in p['opciones']}
    ganadora = max(conteo, key=conteo.get) if votos[id] else None
    return {"pregunta": p['titulo'], "votos": conteo, "ganadora": ganadora, "total": len(votos[id])}


cfg = uvicorn.Config(app_votos, host="127.0.0.1", port=8006, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)

B = "http://127.0.0.1:8006"
p = requests.post(f"{B}/preguntas", json={"titulo": "Dove facciamo il team building?",
    "opciones": ["Madrid", "Barcellona", "Online"]}).json()
print("Domanda:", p['titulo'])

for votante, opcion in [("Ana","Madrid"),("Carlos","Barcellona"),("Lucía","Madrid"),("Pedro","Online")]:
    r = requests.post(f"{B}/preguntas/1/votar", json={"opcion": opcion, "votante": votante})
    print(f"  {votante} → {opcion}: {r.status_code}")

# Doppio voto
r = requests.post(f"{B}/preguntas/1/votar", json={"opcion": "Online", "votante": "Ana"})
print(f"  Ana vota di nuovo: {r.status_code} — {r.json()['detail']}")

print("\nRisultato:", requests.get(f"{B}/preguntas/1/resultado").json())
```
</details>

---
## Riepilogo della lezione

```python
from fastapi import FastAPI, HTTPException, Path, Query
from pydantic import BaseModel, Field

app = FastAPI()

class Item(BaseModel):
    nombre: str = Field(min_length=1)
    precio: float = Field(gt=0)

# Path param con validazione
@app.get("/items/{id}")
def get(id: int = Path(ge=1)):
    ...

# Query params con default
@app.get("/items")
def listar(skip: int = 0, limit: int = Query(10, le=100)):
    ...

# Body JSON → Pydantic valida automaticamente
@app.post("/items", status_code=201)
def crear(item: Item):
    ...

# Errore HTTP
raise HTTPException(status_code=404, detail="Non trovato")

# Lanciare:
# uvicorn main:app --reload
# Docs: http://localhost:8000/docs
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 4** colleghiamo FastAPI a un database reale usando SQLAlchemy: persistenza dei dati, modelli ORM e operazioni CRUD che sopravvivono al riavvio del server.